# Multimodal Explainable AI for Early Diabetes Prediction
## Full Pipeline — Phases 1–5

| Phase | Description |
|---|---|
| 1 | Data loading, preprocessing, feature engineering |
| 2 | Synthetic clinical notes + ClinicalBERT embeddings |
| 3 | Baseline models + Early/Late Fusion + evaluation |
| 4 | SHAP explainability + LLM recommendations |
| 5 | Publication-ready figures and tables |

> **Dataset:** Diabetes 130-US Hospitals 1999–2008 (UCI, ID=296)  
> **Reproducibility:** `RANDOM_SEED = 42` throughout.

## 1. Imports & global config

All libraries for the entire pipeline imported here once.

In [2]:
import os, json, time, warnings, pickle, shutil, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm

from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.calibration import calibration_curve
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from groq import Groq, RateLimitError

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
import shap

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

plt.rcParams.update({
    "figure.facecolor":"white","axes.facecolor":"#F8F9FB",
    "axes.grid":True,"grid.alpha":0.4,
    "axes.spines.top":False,"axes.spines.right":False,"font.size":11,
})
PALETTE = {
    "neg":"#4A90D9","pos":"#E05C5C","neu":"#7F77DD",
    "lr":"#4A90D9","rf":"#56B87A","xgb":"#E0AA00",
    "early":"#E05C5C","late":"#9B59B6","tab":"#7F77DD",
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs("data", exist_ok=True)
os.makedirs("data/models", exist_ok=True)
os.makedirs("data/figures", exist_ok=True)
os.makedirs("data/paper", exist_ok=True)
print(f"Ready. RANDOM_SEED={RANDOM_SEED} | device={device}")

KeyboardInterrupt: 

## 2. Google Drive — mount & restore

Run at the start of **every new Colab session** to restore saved artefacts.

In [4]:
from google.colab import drive, userdata
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/diabetes_project/data"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/models", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/paper", exist_ok=True)

RESTORE_FILES = [
    "scaler.pkl","feature_names.json","diabetes_clean.parquet",
    "X_train.npy","X_val.npy","X_test.npy",
    "y_train.npy","y_val.npy","y_test.npy",
    "text_embeddings.npy","synthetic_notes_final.csv",
    "synthetic_notes.csv","notes_checkpoint.csv",
    "results_summary.csv","shap_values_xgb.npy",
    "recommendations_sample.csv",
]
for fname in RESTORE_FILES:
    src, dst = f"{DRIVE_DIR}/{fname}", f"data/{fname}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst); print(f"  Restored: {fname}")

for mname in ["xgboost.json","logistic_regression.pkl",
              "random_forest.pkl","early_fusion.pt","late_fusion.pt"]:
    src, dst = f"{DRIVE_DIR}/models/{mname}", f"data/models/{mname}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst); print(f"Restored model: {mname}")

print("Restore complete.")

Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: 'data/scaler.pkl'

---
# Phase 4 — SHAP Explainability & LLM Recommendations

## 35. SHAP values — cached

In [ ]:
SHAP_CACHE = "data/shap_values_xgb.npy"
DRIVE_SHAP = f"{DRIVE_DIR}/shap_values_xgb.npy"
X_test_sample = X_test[:1000]
y_test_sample = y_test[:1000]

if os.path.exists(SHAP_CACHE):
    xgb_shap_values=np.load(SHAP_CACHE)
    print(f"SHAP loaded from cache: {xgb_shap_values.shape}")
elif os.path.exists(DRIVE_SHAP):
    shutil.copy2(DRIVE_SHAP,SHAP_CACHE)
    xgb_shap_values=np.load(SHAP_CACHE)
    print(f"SHAP restored from Drive: {xgb_shap_values.shape}")
else:
    print("Computing SHAP values ...")
    background=X_train[np.random.choice(len(X_train),500,replace=False)]
    explainer=shap.TreeExplainer(xgb_model,background)
    xgb_shap_values=explainer.shap_values(X_test_sample)
    np.save(SHAP_CACHE,xgb_shap_values)
    shutil.copy2(SHAP_CACHE,DRIVE_SHAP)
    print(f"SHAP computed and cached: {xgb_shap_values.shape}")

mean_abs_shap=np.abs(xgb_shap_values).mean(axis=0)
importance_df=pd.DataFrame({"feature":FEATURE_COLS,"mean_abs_shap":mean_abs_shap})
importance_df=importance_df.sort_values("mean_abs_shap",ascending=False)
print("Top 5:"); print(importance_df.head(5).to_string(index=False))

## 36. SHAP global summary

In [ ]:
plt.figure(figsize=(10,8))
shap.summary_plot(xgb_shap_values,X_test_sample,feature_names=FEATURE_COLS,max_display=15,show=False)
plt.title("SHAP Summary — XGBoost (Top 15 Features)",fontweight="bold",fontsize=12)
plt.tight_layout()
plt.savefig("data/figures/shap_summary.png",dpi=150,bbox_inches="tight")
plt.show()

## 37. SHAP waterfall — patient level

In [ ]:
hr_idx=np.where(y_test_sample==1)[0][0]
lr_idx=np.where(y_test_sample==0)[0][0]
fig,axes=plt.subplots(1,2,figsize=(18,8))
for ax,pidx,label,tc in [(axes[0],hr_idx,"HIGH RISK",PALETTE["pos"]),
                          (axes[1],lr_idx,"LOW RISK", PALETTE["neg"])]:
    sv=xgb_shap_values[pidx]; fv=X_test_sample[pidx]
    risk=float(xgb_model.predict_proba(X_test_sample[pidx:pidx+1])[:,1][0])
    order=np.argsort(np.abs(sv))[::-1][:10]
    names=[FEATURE_COLS[i] for i in order]; vals=sv[order]; fvals=fv[order]
    colors=[PALETTE["pos"] if v>0 else PALETTE["neg"] for v in vals]
    ax.barh(range(10),vals[::-1],color=colors[::-1],edgecolor="white")
    ax.set_yticks(range(10))
    ax.set_yticklabels([f"{n}={fvals[9-i]:.2f}" for i,n in enumerate(names[::-1])],fontsize=8)
    ax.axvline(0,color="black",lw=0.8); ax.set_xlabel("SHAP Value")
    ax.set_title(f"[{label}]  Risk: {risk:.1%}",fontweight="bold",color=tc,fontsize=11)
plt.suptitle("Patient-Level SHAP Explanations",fontsize=12,fontweight="bold")
plt.tight_layout()
plt.savefig("data/figures/shap_waterfall.png",dpi=150,bbox_inches="tight")
plt.show()

## 38. LLM recommendations — cached

In [ ]:
LOCAL_REC = "data/recommendations_sample.csv"
DRIVE_REC = f"{DRIVE_DIR}/recommendations_sample.csv"

if os.path.exists(LOCAL_REC):
    rec_df=pd.read_csv(LOCAL_REC)
    print(f"Loaded from local cache: {len(rec_df):,} recommendations")
elif os.path.exists(DRIVE_REC):
    shutil.copy2(DRIVE_REC,LOCAL_REC)
    rec_df=pd.read_csv(LOCAL_REC)
    print(f"Restored from Drive: {len(rec_df):,} recommendations")
else:
    REC_SYSTEM = (
        "You are a clinical decision support assistant. "
        "Based on the patient risk and key factors, give a concise clinical "
        "recommendation (3-4 sentences). Specific, actionable, medical language. No bullets."
    )
    def build_rec_prompt(fv,sv,feat_names,risk,top_n=3):
        pos_idx=np.argsort(sv)[::-1][:top_n]
        pos_txt=", ".join(f"{feat_names[i]}={fv[i]:.2f}(SHAP={sv[i]:+.3f})"
                          for i in pos_idx if sv[i]>0)
        neg_idx=np.argsort(sv)[:top_n]
        neg_txt=", ".join(f"{feat_names[i]}={fv[i]:.2f}(SHAP={sv[i]:+.3f})"
                          for i in neg_idx if sv[i]<0)
        return (f"Patient readmission risk: {risk:.1%}\n"
                f"Risk-increasing: {pos_txt or 'none'}\n"
                f"Risk-reducing: {neg_txt or 'none'}\n"
                "Clinical recommendation:")

    N_PER_CLASS=50
    hr_indices=np.where(y_test_sample==1)[0][:N_PER_CLASS]
    lr_indices=np.where(y_test_sample==0)[0][:N_PER_CLASS]
    rec_indices=np.concatenate([hr_indices,lr_indices])
    recommendations=[]; print(f"Generating {len(rec_indices)} recommendations ...")

    for i,idx in enumerate(rec_indices):
        risk=float(xgb_model.predict_proba(X_test_sample[idx:idx+1])[:,1][0])
        prompt=build_rec_prompt(X_test_sample[idx],xgb_shap_values[idx],FEATURE_COLS,risk)
        rec_text=None
        for attempt in range(3):
            try:
                resp=client.chat.completions.create(
                    model="llama-3.1-8b-instant",max_tokens=150,temperature=0.2,
                    messages=[{"role":"system","content":REC_SYSTEM},
                              {"role":"user","content":prompt}])
                rec_text=resp.choices[0].message.content.strip(); break
            except RateLimitError: time.sleep(2**attempt)
            except Exception: break
        recommendations.append({"patient_idx":int(idx),"true_label":int(y_test_sample[idx]),
                                 "predicted_risk":round(risk,4),"recommendation":rec_text or "[FAILED]"})
        if (i+1)%20==0: print(f"  {i+1}/{len(rec_indices)}")
        time.sleep(0.05)

    rec_df=pd.DataFrame(recommendations)
    rec_df.to_csv(LOCAL_REC,index=False)
    shutil.copy2(LOCAL_REC,DRIVE_REC)
    print(f"Generated and saved: {len(rec_df):,}")

rec_df["word_count"]=rec_df["recommendation"].str.split().str.len()
print("\nStats by class:")
print(rec_df.groupby("true_label")[["predicted_risk","word_count"]].mean().round(3))

---
# Phase 5 — Publication-Ready Figures & Tables